In [1]:
#!/usr/bin/env python3
import os
import glob
import re
import numpy as np
import pandas as pd
import h5py
import matplotlib
import matplotlib.pyplot as plt
import logging

from bm3d import bm3d
from bm4d import bm4d

# ==========================================
# 0. CONFIGURATION 
# ==========================================
base_dir = "/pscratch/sd/k/kberard/SCGSR/Data/diamond_1x1x1_bfd/density_data/vmc_J2"
ref_path = os.path.join(base_dir, "density_tot_ref_mean.h5")
dft_path = '/global/u2/k/kberard/SCGSR/Research/Diamond/Data/density_tot_ref.h5'

# ==========================================
# 1. UNIFIED MATH & UTILITY FUNCTIONS
# ==========================================
def D_JS(p1, p2, tol=1e-16):
    """Calculates Jensen-Shannon Divergence."""
    p1 = p1 / (np.sum(p1) + 1e-16)
    p2 = p2 / (np.sum(p2) + 1e-16)
    pm = (p1 + p2) / 2
    p1_nonzero, p2_nonzero = np.abs(p1) > tol, np.abs(p2) > tol
    
    d = 0.5 * (
        (np.abs(p1[p1_nonzero]) * np.log(np.abs(p1[p1_nonzero]) / np.abs(pm[p1_nonzero]))).sum() + 
        (np.abs(p2[p2_nonzero]) * np.log(np.abs(p2[p2_nonzero]) / np.abs(pm[p2_nonzero]))).sum()
    )
    return d / np.log(2)

def transform(density, density_ref, transform_type='residual_noise'):
    if transform_type == 'residual_noise':
        return (density - density_ref) / (np.sqrt(np.abs(density_ref)) + 1e-12)
    return density

def inverse_transform(density_trans, density_ref, transform_type='residual_noise'):
    if transform_type == 'residual_noise':
        return density_ref + (np.sqrt(np.abs(density_ref)) * density_trans)
    return density_trans

def encode_voxel_to_rgb_global(vol_3d):
    v_min, v_max = float(vol_3d.min()), float(vol_3d.max())
    if v_max == v_min: v_max = v_min + 1e-6
    normed = (vol_3d - v_min) / (v_max - v_min)
    return np.stack([normed]*3, axis=-1).astype(np.float32), v_min, v_max

def decode_rgb_to_voxel_global(rgb_volume, v_min, v_max):
    return rgb_volume[:, :, :, 0] * (v_max - v_min) + v_min

def evaluate_and_enforce(denoised_d, ref_d):
    denoised_d = np.maximum(denoised_d, 0.0)
    denoised_d = denoised_d * (8.0 / (np.sum(denoised_d) + 1e-16))
    return D_JS(denoised_d, ref_d)

# ==========================================
# 2. INFERENCE DISPATCHER
# ==========================================
def run_bm_with_sigma(test_d, ref_d_dft, model_name, sigma):
    """Runs the algorithm with an explicit sigma value."""
    trans_d = transform(test_d, ref_d_dft, 'residual_noise')
    
    if model_name == 'bm3d':
        rgb_vol, v_min, v_max = encode_voxel_to_rgb_global(trans_d)
        denoised_rgb = np.zeros_like(rgb_vol)
        for i in range(trans_d.shape[0]):
            denoised_gray = bm3d(rgb_vol[i, :, :, 0], sigma_psd=sigma)
            denoised_rgb[i] = np.stack([denoised_gray]*3, axis=-1)
        denoised_trans = decode_rgb_to_voxel_global(denoised_rgb, v_min, v_max)

    elif model_name == 'bm4d':
        denoised_trans = bm4d(trans_d, sigma_psd=sigma)

    return inverse_transform(denoised_trans, ref_d_dft, 'residual_noise')

# ==========================================
# 3. MAIN EXECUTION PIPELINE
# ==========================================
def main():
    print("Loading Base Data & DFT Reference...")
    with h5py.File(ref_path, 'r') as file:
        ref_d_mean = file['density'][:]
        ref_d_mean = ref_d_mean * (8.0 / np.sum(ref_d_mean))

    with h5py.File(dft_path, 'r') as file:
        dft_d = file['density'][:]
        dft_d = dft_d * (8.0 / np.sum(dft_d))
        
    noisy_files = sorted(glob.glob(os.path.join(base_dir, "density_tot_vmc_mean*.h5")))
    
    # Extract all sample numbers and find the exact median
    sample_nums = sorted([int(re.search(r"(\d+)\.h5$", f).group(1)) for f in noisy_files if re.search(r"(\d+)\.h5$", f)])
    mid_sample = sample_nums[len(sample_nums) // 2]
    target_file = [f for f in noisy_files if f"{mid_sample:010d}.h5" in f or f"_{mid_sample}.h5" in f][0]
    
    print(f"Selected median sample count: {mid_sample}")
    print(f"Loading file: {target_file}")
    
    with h5py.File(target_file, 'r') as file:
        test_d = file['density'][:]

    # Baseline JSD for reference
    jsd_baseline = evaluate_and_enforce(test_d, ref_d_mean)
    print(f"Baseline Noisy VMC JSD: {jsd_baseline:.6e}\n")

    # Define the Sigma search grid (from 1e-8 to 1.0)
    sigmas_to_test = np.logspace(-8, 0, 9)
    
    results = {
        'Sigma': sigmas_to_test,
        'BM3D_JSD': [],
        'BM4D_JSD': []
    }

    print("=== Commencing Parameter Sweep ===")
    for sigma in sigmas_to_test:
        print(f"\nTesting Sigma = {sigma:.1e}")
        
        # Test BM3D
        denoised_3d = run_bm_with_sigma(test_d, dft_d, 'bm3d', sigma)
        jsd_3d = evaluate_and_enforce(denoised_3d, ref_d_mean)
        results['BM3D_JSD'].append(jsd_3d)
        print(f"   ↳ BM3D JSD: {jsd_3d:.6e}")
        
        # Test BM4D
        denoised_4d = run_bm_with_sigma(test_d, dft_d, 'bm4d', sigma)
        jsd_4d = evaluate_and_enforce(denoised_4d, ref_d_mean)
        results['BM4D_JSD'].append(jsd_4d)
        print(f"   ↳ BM4D JSD: {jsd_4d:.6e}")

    # ==========================================
    # 4. RESULTS AND PLOTTING
    # ==========================================
    df = pd.DataFrame(results)
    df.to_csv("Sigma_Parameter_Sweep.csv", index=False)
    
    best_bm3d_sigma = df.loc[df['BM3D_JSD'].idxmin(), 'Sigma']
    best_bm4d_sigma = df.loc[df['BM4D_JSD'].idxmin(), 'Sigma']
    
    print("\n" + "="*40)
    print("PARAMETER SWEEP COMPLETE")
    print(f"Optimal BM3D Sigma: {best_bm3d_sigma:.1e} (JSD: {df['BM3D_JSD'].min():.6e})")
    print(f"Optimal BM4D Sigma: {best_bm4d_sigma:.1e} (JSD: {df['BM4D_JSD'].min():.6e})")
    print("="*40 + "\n")

    # Plot the Loss Landscape
    matplotlib.use('Agg')
    plt.rcParams.update({
        'font.size': 14, 'font.family': 'serif', 'axes.labelsize': 16,
        'axes.linewidth': 1.5, 'xtick.major.size': 8, 'xtick.major.width': 1.5,
        'ytick.major.size': 8, 'ytick.major.width': 1.5,
        'legend.frameon': True, 'legend.edgecolor': 'black'
    })

    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.plot(df['Sigma'], df['BM3D_JSD'], marker='s', linestyle='--', color='blue', label='BM3D', linewidth=2.5, markersize=8)
    ax.plot(df['Sigma'], df['BM4D_JSD'], marker='D', linestyle='-', color='green', label='BM4D', linewidth=2.5, markersize=8)
    ax.axhline(jsd_baseline, color='red', linestyle=':', label='Noisy VMC Baseline', linewidth=2)
    
    # Mark the optimal points
    ax.scatter([best_bm3d_sigma], [df['BM3D_JSD'].min()], color='black', s=150, zorder=5, marker='*')
    ax.scatter([best_bm4d_sigma], [df['BM4D_JSD'].min()], color='black', s=150, zorder=5, marker='*', label='Optimal Minimum')

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Sigma Parameter', fontweight='bold')
    ax.set_ylabel(r'$D_{JS}$ (Jensen-Shannon Divergence)', fontweight='bold')
    ax.set_title(f'Sigma Optimization Landscape (Sample Size: {mid_sample})', fontweight='bold')
    
    ax.legend(loc='best')
    plt.tight_layout()
    plt.savefig("Sigma_Optimization_Landscape.png", dpi=300)
    plt.savefig("Sigma_Optimization_Landscape.pdf")
    print("Optimization plots saved.")

if __name__ == "__main__":
    main()

Loading Base Data & DFT Reference...
Selected median sample count: 10321920
Loading file: /pscratch/sd/k/kberard/SCGSR/Data/diamond_1x1x1_bfd/density_data/vmc_J2/density_tot_vmc_mean_0010321920.h5
Baseline Noisy VMC JSD: 6.003742e-04

=== Commencing Parameter Sweep ===

Testing Sigma = 1.0e-08
   ↳ BM3D JSD: 6.003604e-04
   ↳ BM4D JSD: 5.626061e-04

Testing Sigma = 1.0e-07
   ↳ BM3D JSD: 6.003604e-04
   ↳ BM4D JSD: 5.626061e-04

Testing Sigma = 1.0e-06
   ↳ BM3D JSD: 6.003604e-04
   ↳ BM4D JSD: 5.626063e-04

Testing Sigma = 1.0e-05
   ↳ BM3D JSD: 6.003604e-04
   ↳ BM4D JSD: 5.625829e-04

Testing Sigma = 1.0e-04
   ↳ BM3D JSD: 6.003604e-04
   ↳ BM4D JSD: 5.413767e-04

Testing Sigma = 1.0e-03
   ↳ BM3D JSD: 6.003011e-04
   ↳ BM4D JSD: 5.629930e-06

Testing Sigma = 1.0e-02
   ↳ BM3D JSD: 5.947236e-04
   ↳ BM4D JSD: 8.063109e-05

Testing Sigma = 1.0e-01
   ↳ BM3D JSD: 1.191862e-05
   ↳ BM4D JSD: 1.259048e-04

Testing Sigma = 1.0e+00
   ↳ BM3D JSD: 1.477623e-05
   ↳ BM4D JSD: 1.267153e-04



In [3]:
from skimage import data
from skimage.restoration import estimate_sigma


In [4]:
#!/usr/bin/env python3
import os
import glob
import re
import numpy as np
import h5py
from skimage.restoration import estimate_sigma

from bm3d import bm3d
from bm4d import bm4d

# ==========================================
# 0. CONFIGURATION 
# ==========================================
base_dir = "/pscratch/sd/k/kberard/SCGSR/Data/diamond_1x1x1_bfd/density_data/vmc_J2"
ref_path = os.path.join(base_dir, "density_tot_ref_mean.h5")
dft_path = '/global/u2/k/kberard/SCGSR/Research/Diamond/Data/density_tot_ref.h5'

# ==========================================
# 1. UNIFIED MATH & UTILITY FUNCTIONS
# ==========================================
def D_JS(p1, p2, tol=1e-16):
    """Calculates Jensen-Shannon Divergence."""
    p1 = p1 / (np.sum(p1) + 1e-16)
    p2 = p2 / (np.sum(p2) + 1e-16)
    pm = (p1 + p2) / 2
    p1_nonzero, p2_nonzero = np.abs(p1) > tol, np.abs(p2) > tol
    
    d = 0.5 * (
        (np.abs(p1[p1_nonzero]) * np.log(np.abs(p1[p1_nonzero]) / np.abs(pm[p1_nonzero]))).sum() + 
        (np.abs(p2[p2_nonzero]) * np.log(np.abs(p2[p2_nonzero]) / np.abs(pm[p2_nonzero]))).sum()
    )
    return d / np.log(2)

def transform(density, density_ref, transform_type='residual_noise'):
    if transform_type == 'residual_noise':
        return (density - density_ref) / (np.sqrt(np.abs(density_ref)) + 1e-12)
    return density

def inverse_transform(density_trans, density_ref, transform_type='residual_noise'):
    if transform_type == 'residual_noise':
        return density_ref + (np.sqrt(np.abs(density_ref)) * density_trans)
    return density_trans

def encode_voxel_to_rgb_global(vol_3d):
    v_min, v_max = float(vol_3d.min()), float(vol_3d.max())
    if v_max == v_min: v_max = v_min + 1e-6
    normed = (vol_3d - v_min) / (v_max - v_min)
    return np.stack([normed]*3, axis=-1).astype(np.float32), v_min, v_max

def decode_rgb_to_voxel_global(rgb_volume, v_min, v_max):
    return rgb_volume[:, :, :, 0] * (v_max - v_min) + v_min

def evaluate_and_enforce(denoised_d, ref_d):
    denoised_d = np.maximum(denoised_d, 0.0)
    denoised_d = denoised_d * (8.0 / (np.sum(denoised_d) + 1e-16))
    return D_JS(denoised_d, ref_d)

# ==========================================
# 2. INFERENCE DISPATCHER
# ==========================================
def run_bm_with_sigma(test_d, ref_d_dft, model_name, sigma):
    """Runs the algorithm with an explicit sigma value."""
    trans_d = transform(test_d, ref_d_dft, 'residual_noise')
    
    if model_name == 'bm3d':
        rgb_vol, v_min, v_max = encode_voxel_to_rgb_global(trans_d)
        denoised_rgb = np.zeros_like(rgb_vol)
        for i in range(trans_d.shape[0]):
            denoised_gray = bm3d(rgb_vol[i, :, :, 0], sigma_psd=sigma)
            denoised_rgb[i] = np.stack([denoised_gray]*3, axis=-1)
        denoised_trans = decode_rgb_to_voxel_global(denoised_rgb, v_min, v_max)

    elif model_name == 'bm4d':
        denoised_trans = bm4d(trans_d, sigma_psd=sigma)

    return inverse_transform(denoised_trans, ref_d_dft, 'residual_noise')

# ==========================================
# 3. MAIN EXECUTION PIPELINE
# ==========================================
def main():
    print("Loading Base Data & DFT Reference...")
    with h5py.File(ref_path, 'r') as file:
        ref_d_mean = file['density'][:]
        ref_d_mean = ref_d_mean * (8.0 / np.sum(ref_d_mean))

    with h5py.File(dft_path, 'r') as file:
        dft_d = file['density'][:]
        dft_d = dft_d * (8.0 / np.sum(dft_d))
        
    noisy_files = sorted(glob.glob(os.path.join(base_dir, "density_tot_vmc_mean*.h5")))
    
    # Target specific sample
    target_sample = 6881280
    try:
        target_file = [f for f in noisy_files if f"{target_sample:010d}.h5" in f or f"_{target_sample}.h5" in f][0]
    except IndexError:
        raise FileNotFoundError(f"Could not find a file matching sample {target_sample} in {base_dir}")
    
    print(f"Selected sample count: {target_sample}")
    print(f"Loading file: {target_file}")
    
    with h5py.File(target_file, 'r') as file:
        test_d = file['density'][:]

    # Baseline JSD for reference
    jsd_baseline = evaluate_and_enforce(test_d, ref_d_mean)
    print(f"Baseline Noisy VMC JSD: {jsd_baseline:.6e}\n")

    # ==========================================
    # 4. AUTOMATIC SIGMA ESTIMATION & DENOISING
    # ==========================================
    print("=== Estimating Sigma ===")
    # We estimate sigma on the transformed data since that is the domain being denoised
    trans_d = transform(test_d, dft_d, 'residual_noise')
    
    # estimate_sigma by default treats the array as a single-channel N-dimensional image
    estimated_sigma = estimate_sigma(trans_d)
    print(f"skimage estimated Sigma: {estimated_sigma:.6e}\n")

    print("=== Running Denoising Models ===")
    # Run BM3D
    denoised_3d = run_bm_with_sigma(test_d, dft_d, 'bm3d', estimated_sigma)
    jsd_3d = evaluate_and_enforce(denoised_3d, ref_d_mean)
    print(f"  ↳ BM3D JSD: {jsd_3d:.6e}")
    
    # Run BM4D
    denoised_4d = run_bm_with_sigma(test_d, dft_d, 'bm4d', estimated_sigma)
    jsd_4d = evaluate_and_enforce(denoised_4d, ref_d_mean)
    print(f"  ↳ BM4D JSD: {jsd_4d:.6e}")

    print("\n" + "="*40)
    print("EVALUATION COMPLETE")
    print(f"Sample: {target_sample}")
    print(f"Applied Sigma: {estimated_sigma:.6e}")
    print(f"Final BM3D JSD: {jsd_3d:.6e}")
    print(f"Final BM4D JSD: {jsd_4d:.6e}")
    print("="*40 + "\n")

if __name__ == "__main__":
    main()

Loading Base Data & DFT Reference...
Selected sample count: 6881280
Loading file: /pscratch/sd/k/kberard/SCGSR/Data/diamond_1x1x1_bfd/density_data/vmc_J2/density_tot_vmc_mean_0006881280.h5
Baseline Noisy VMC JSD: 9.014070e-04

=== Estimating Sigma ===
skimage estimated Sigma: 3.834905e-04

=== Running Denoising Models ===
  ↳ BM3D JSD: 9.013754e-04
  ↳ BM4D JSD: 8.223303e-06

EVALUATION COMPLETE
Sample: 6881280
Applied Sigma: 3.834905e-04
Final BM3D JSD: 9.013754e-04
Final BM4D JSD: 8.223303e-06



In [5]:
#!/usr/bin/env python3
import os
import glob
import re
import numpy as np
import h5py
from skimage.restoration import estimate_sigma

from bm3d import bm3d
from bm4d import bm4d

# ==========================================
# 0. CONFIGURATION 
# ==========================================
base_dir = "/pscratch/sd/k/kberard/SCGSR/Data/diamond_1x1x1_bfd/density_data/vmc_J2"
ref_path = os.path.join(base_dir, "density_tot_ref_mean.h5")
dft_path = '/global/u2/k/kberard/SCGSR/Research/Diamond/Data/density_tot_ref.h5'

# ==========================================
# 1. UNIFIED MATH & UTILITY FUNCTIONS
# ==========================================
def D_JS(p1, p2, tol=1e-16):
    """Calculates Jensen-Shannon Divergence."""
    p1 = p1 / (np.sum(p1) + 1e-16)
    p2 = p2 / (np.sum(p2) + 1e-16)
    pm = (p1 + p2) / 2
    p1_nonzero, p2_nonzero = np.abs(p1) > tol, np.abs(p2) > tol
    
    d = 0.5 * (
        (np.abs(p1[p1_nonzero]) * np.log(np.abs(p1[p1_nonzero]) / np.abs(pm[p1_nonzero]))).sum() + 
        (np.abs(p2[p2_nonzero]) * np.log(np.abs(p2[p2_nonzero]) / np.abs(pm[p2_nonzero]))).sum()
    )
    return d / np.log(2)

def transform(density, density_ref, transform_type='residual_noise'):
    if transform_type == 'residual_noise':
        return (density - density_ref) / (np.sqrt(np.abs(density_ref)) + 1e-12)
    return density

def inverse_transform(density_trans, density_ref, transform_type='residual_noise'):
    if transform_type == 'residual_noise':
        return density_ref + (np.sqrt(np.abs(density_ref)) * density_trans)
    return density_trans

def encode_voxel_to_rgb_global(vol_3d):
    v_min, v_max = float(vol_3d.min()), float(vol_3d.max())
    if v_max == v_min: v_max = v_min + 1e-6
    normed = (vol_3d - v_min) / (v_max - v_min)
    return np.stack([normed]*3, axis=-1).astype(np.float32), v_min, v_max

def decode_rgb_to_voxel_global(rgb_volume, v_min, v_max):
    return rgb_volume[:, :, :, 0] * (v_max - v_min) + v_min

def evaluate_and_enforce(denoised_d, ref_d):
    denoised_d = np.maximum(denoised_d, 0.0)
    denoised_d = denoised_d * (8.0 / (np.sum(denoised_d) + 1e-16))
    return D_JS(denoised_d, ref_d)

# ==========================================
# 2. INFERENCE DISPATCHER
# ==========================================
def run_bm_with_sigma(test_d, ref_d_dft, model_name, sigma, use_transform=False):
    """Runs the algorithm with an explicit sigma value, optionally bypassing transformation."""
    if use_transform:
        trans_d = transform(test_d, ref_d_dft, 'residual_noise')
    else:
        trans_d = test_d
    
    if model_name == 'bm3d':
        rgb_vol, v_min, v_max = encode_voxel_to_rgb_global(trans_d)
        denoised_rgb = np.zeros_like(rgb_vol)
        for i in range(trans_d.shape[0]):
            denoised_gray = bm3d(rgb_vol[i, :, :, 0], sigma_psd=sigma)
            denoised_rgb[i] = np.stack([denoised_gray]*3, axis=-1)
        denoised_trans = decode_rgb_to_voxel_global(denoised_rgb, v_min, v_max)

    elif model_name == 'bm4d':
        denoised_trans = bm4d(trans_d, sigma_psd=sigma)

    if use_transform:
        return inverse_transform(denoised_trans, ref_d_dft, 'residual_noise')
    return denoised_trans

# ==========================================
# 3. MAIN EXECUTION PIPELINE
# ==========================================
def main():
    print("Loading Base Data & DFT Reference...")
    with h5py.File(ref_path, 'r') as file:
        ref_d_mean = file['density'][:]
        ref_d_mean = ref_d_mean * (8.0 / np.sum(ref_d_mean))

    with h5py.File(dft_path, 'r') as file:
        dft_d = file['density'][:]
        dft_d = dft_d * (8.0 / np.sum(dft_d))
        
    noisy_files = sorted(glob.glob(os.path.join(base_dir, "density_tot_vmc_mean*.h5")))
    
    # Target specific sample
    target_sample = 6881280
    try:
        target_file = [f for f in noisy_files if f"{target_sample:010d}.h5" in f or f"_{target_sample}.h5" in f][0]
    except IndexError:
        raise FileNotFoundError(f"Could not find a file matching sample {target_sample} in {base_dir}")
    
    print(f"Selected sample count: {target_sample}")
    print(f"Loading file: {target_file}")
    
    with h5py.File(target_file, 'r') as file:
        test_d = file['density'][:]

    # Baseline JSD for reference
    jsd_baseline = evaluate_and_enforce(test_d, ref_d_mean)
    print(f"Baseline Noisy VMC JSD: {jsd_baseline:.6e}\n")

    # ==========================================
    # 4. AUTOMATIC SIGMA ESTIMATION & DENOISING
    # ==========================================
    print("=== Estimating Sigma ===")
    # Calculate transformed data for comparative reference
    trans_d = transform(test_d, dft_d, 'residual_noise')
    sigma_transformed = estimate_sigma(trans_d)
    
    # Calculate raw sigma directly on the non-transformed test data
    sigma_raw = estimate_sigma(test_d)
    
    print(f"skimage estimated Sigma (Transformed):     {sigma_transformed:.6e}")
    print(f"skimage estimated Sigma (Non-Transformed): {sigma_raw:.6e}\n")

    print("=== Running Denoising Models on Non-Transformed Data ===")
    # Run BM3D directly on raw data
    denoised_3d = run_bm_with_sigma(test_d, dft_d, 'bm3d', sigma_raw, use_transform=False)
    jsd_3d = evaluate_and_enforce(denoised_3d, ref_d_mean)
    print(f"  ↳ BM3D JSD (Non-Transformed): {jsd_3d:.6e}")
    
    # Run BM4D directly on raw data
    denoised_4d = run_bm_with_sigma(test_d, dft_d, 'bm4d', sigma_raw, use_transform=False)
    jsd_4d = evaluate_and_enforce(denoised_4d, ref_d_mean)
    print(f"  ↳ BM4D JSD (Non-Transformed): {jsd_4d:.6e}")

    print("\n" + "="*40)
    print("EVALUATION COMPLETE")
    print(f"Sample: {target_sample}")
    print(f"Applied Raw Sigma: {sigma_raw:.6e}")
    print(f"Final BM3D JSD: {jsd_3d:.6e}")
    print(f"Final BM4D JSD: {jsd_4d:.6e}")
    print("="*40 + "\n")

if __name__ == "__main__":
    main()

Loading Base Data & DFT Reference...
Selected sample count: 6881280
Loading file: /pscratch/sd/k/kberard/SCGSR/Data/diamond_1x1x1_bfd/density_data/vmc_J2/density_tot_vmc_mean_0006881280.h5
Baseline Noisy VMC JSD: 9.014070e-04

=== Estimating Sigma ===
skimage estimated Sigma (Transformed):     3.834905e-04
skimage estimated Sigma (Non-Transformed): 1.622138e-06

=== Running Denoising Models on Non-Transformed Data ===
  ↳ BM3D JSD (Non-Transformed): 9.013596e-04
  ↳ BM4D JSD (Non-Transformed): 9.262873e-05

EVALUATION COMPLETE
Sample: 6881280
Applied Raw Sigma: 1.622138e-06
Final BM3D JSD: 9.013596e-04
Final BM4D JSD: 9.262873e-05

